# Forecasting Sri Lanka Telecom Stock Prices Using Ensemble Learning
### A Comparative Study of Random Forest, XGBoost and Stacking Models

**Student name / ID:** ______________________ *(to be completed by the student)*  
**Dataset:** `TeleCom_2009_2023_Daily.csv` (daily records, 2009-2023)

> This is a *historical back-test* for a university assignment. It is **not** a live trading system and **not** financial advice.

**Contents:** 0 Setup · 1 Introduction · 2 Dataset and EDA · 3 Preprocessing and leakage prevention · 4 Persistence baseline · 5 Random Forest · 6 XGBoost · 7 Stacking · 8 Return-based vs raw-price · 9 Evaluation and visualisations · 10 Interpretation · 11 Limitations and conclusions · 12 AI usage disclosure · 13 Appendix

**How it works.** The notebook does **not retrain** the models. It loads the models and results saved in Phases 2-3 (`models/`, `results/`, `data/processed/`). Every table, plot and number in the interpretation text is computed from those files when you run it (**Kernel -> Restart & Run All**). The only typed-in numbers are in the authoritative-run check in Section 0.

## 0. Setup and environment check

In [ ]:
import sys, json, platform, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from scipy import stats
import sklearn, xgboost
from IPython.display import display, Markdown

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for folder in [here, *here.parents][:5]:
        if (folder / "src" / "ensemble_models.py").exists() and (folder / "models" / "model_registry.json").exists():
            return folder
    raise FileNotFoundError("Open this notebook from inside the project folder (the one containing src/ and models/).")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))
import data_utils as du, preprocessing as pp, evaluation as ev, ensemble_models as em   # em is needed to load saved models
MODELS_DIR, RESULTS_DIR, PROCESSED_DIR = du.MODELS_DIR, du.RESULTS_DIR, du.PROCESSED_DIR

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
LABELS = {"baseline": "Persistence baseline", "random_forest": "Random Forest (Bagging)", "xgboost": "XGBoost (Boosting)", "stacking": "Stacking (RF + XGB -> Ridge)"}
COLORS = {LABELS["baseline"]: "#7f7f7f", LABELS["random_forest"]: "#2ca02c", LABELS["xgboost"]: "#d62728", LABELS["stacking"]: "#1f77b4"}
SPLIT_COLORS = {"train": "#1f77b4", "validation": "#ff7f0e", "test": "#2ca02c"}
print("Project root:", ROOT)

In [ ]:
registry = json.loads((MODELS_DIR / "model_registry.json").read_text())
here = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__, "xgboost": xgboost.__version__}
env = pd.DataFrame({"this notebook": here, "when the models were trained": registry["versions"]})
env["same?"] = [(".".join(str(a).split(".")[:2]) == ".".join(str(b).split(".")[:2])) if i == "python" else a == b
                for i, (a, b) in zip(env.index, zip(env["this notebook"], env["when the models were trained"]))]
display(env); print("Platform:", platform.platform(), "|", platform.machine())
if not env["same?"].all(): print("NOTE: package versions differ from the training run; saved models may fail to load or differ slightly.")

In [ ]:
# AUTHORITATIVE-RUN CHECK. These values were copied (4 d.p.) from the metrics printed on the student's MacBook Air M1
# (Python 3.11.18). They are NOT used for any result below; they only tell you whether the files you are reading are that run.
MAC_RUN = {("Persistence baseline", "validation"): (0.5247, 0.8022, 0.9675), ("Persistence baseline", "test"): (1.3903, 2.4675, 0.9806),
           ("Random Forest (Bagging)", "validation"): (0.5087, 0.7817, 0.9692), ("Random Forest (Bagging)", "test"): (1.4019, 2.4834, 0.9804),
           ("XGBoost (Boosting)", "validation"): (0.5057, 0.7768, 0.9695), ("XGBoost (Boosting)", "test"): (1.4120, 2.5032, 0.9801),
           ("Stacking (RF + XGB -> Ridge)", "validation"): (0.5051, 0.7750, 0.9697), ("Stacking (RF + XGB -> Ridge)", "test"): (1.4068, 2.4886, 0.9803)}
MAC_HP = {"return": {"random_forest": {"max_depth": 15, "min_samples_leaf": 150, "max_features": 1.0, "n_estimators": 300},
                     "xgboost": {"max_depth": 2, "learning_rate": 0.03, "min_child_weight": 1, "n_estimators": 400}, "ridge_alpha": 10.0},
          "rawprice": {"random_forest": {"max_depth": 15, "min_samples_leaf": 5, "max_features": 0.2, "n_estimators": 300},
                       "xgboost": {"max_depth": 3, "learning_rate": 0.03, "min_child_weight": 30, "n_estimators": 400}, "ridge_alpha": 1000.0}}
_r = pd.read_csv(RESULTS_DIR / "metrics_all.csv").query("approach == 'return'").set_index(["model", "split"])
problems = [f"{m}/{s}/{n}: folder {_r.loc[(m, s), n]:.4f} vs Mac {v:.4f}" for (m, s), ref in MAC_RUN.items()
            for n, v in zip(["MAE", "RMSE", "R2"], ref) if abs(_r.loc[(m, s), n] - v) > 6e-5]
problems += [f"hyperparameters ({a}) differ from the Mac run" for a, hp in MAC_HP.items() if registry["hyperparameters"][a] != hp]
if not problems: print("OK: these files match the verified MacBook Air M1 run (metrics to 4 d.p. and all selected hyperparameters).")
else:
    print("NOTE: these files are NOT identical to the verified Mac run:"); [print("  -", p) for p in problems]
    print("The notebook still runs and describes the files you actually have.")

## 1. Introduction and objectives

**Problem.** For each trading observation *t* we use information available **at the close of day *t*** to forecast the closing price of the **next trading observation** *t+1* (the next row of the data, not necessarily the next calendar day).

**Target.** Tree models cannot output prices above those seen in training, and this stock rose far above its training range. So the primary target is the next-observation **log return** and the price is reconstructed:

$$ y_t=\ln\!\left(\frac{P_{t+1}}{P_t}\right),\qquad \widehat P_{t+1}=P_t\,e^{\widehat y_t} $$

All metrics (MAE, RMSE, R²) are computed on the **reconstructed prices in LKR**. Section 8 compares this with a secondary experiment that predicts the raw price.

**Research questions.** (1) Can ensembles beat the **persistence baseline** ("tomorrow = today")? (2) How do **Bagging**, **Boosting** and **Stacking** compare? (3) Does validation performance carry over to the later test period?

| Idea | How it works | Model here |
|---|---|---|
| **Bagging** | Train many models on bootstrap resamples and **average** them (less variance). | Random Forest |
| **Boosting** | Train models **in sequence**, each correcting the previous errors (less bias). | XGBoost |
| **Stacking** | A **meta-model** learns to combine other models' predictions. | RF + XGBoost -> Ridge |

Only the supplied daily CSV is used; hyper-parameters were chosen on validation only; the test set was used once at the end.

## 2. Dataset description and exploratory data analysis
`pp.load_and_clean` reads every column as text first (nothing is guessed), then converts dates, prices, `Vol.` (with `K`/`M` suffixes) and `Change %`. The CSV is only read, never written.

In [ ]:
CSV_PATH = next((p for p in [ROOT / "TeleCom_2009_2023_Daily.csv", ROOT / "data" / "TeleCom_2009_2023_Daily.csv"] if p.exists()), None)
assert CSV_PATH is not None, "Put TeleCom_2009_2023_Daily.csv in the project folder (or data/)."
raw, df = pp.load_and_clean(CSV_PATH)
df["split"] = pp.assign_split(len(df))
print("Raw shape:", raw.shape, "| columns:", list(raw.columns)); display(raw.head(6))
q = pp.quality_report(raw, df)
display(pd.Series({"records": q["rows"], "first date": q["first_date"], "last date": q["last_date"], "duplicate rows": q["duplicate_rows"],
    "duplicate dates": q["duplicate_dates"], "missing Vol. values": q["raw_missing_per_column"]["Vol."],
    "missing values, all other columns": sum(v for k, v in q["raw_missing_per_column"].items() if k != "Vol."), "zero-volume days": q["zero_volume_rows"],
    "High < Low": q["high_below_low"], "High below Open/Close": q["high_below_open_or_close"], "Low above Open/Close": q["low_above_open_or_close"],
    "moves > 10% (log return)": q["abs_log_return_gt_10pct"], "largest gap (days)": q["largest_calendar_gap_days"]}, name="value").to_frame())
meta2 = json.loads((PROCESSED_DIR / "phase2_metadata.json").read_text())
print("Original CSV unchanged since Phase 2 (SHA-256 equal):", pp.sha256(CSV_PATH) == meta2["source_csv_sha256"])

In [ ]:
gap = df["Date"].diff().dt.days
display(df.loc[gap > 7, ["Date"]].assign(days_since_previous=gap[gap > 7].astype(int)).rename(columns={"Date": "first date after the gap"}).reset_index(drop=True))
miss = df.loc[df["Volume_raw"].isna()]
print(f"Missing volumes: {len(miss)} | {miss['Date'].min().date()} to {miss['Date'].max().date()} | all in the training period: {bool((miss['split'] == 'train').all())}")
fig, ax = plt.subplots(figsize=(9, 2.6)); df["Date"].dt.year.value_counts().sort_index().plot.bar(ax=ax, color="#4c72b0")
ax.set_title("Trading observations per year (2009 and 2023 are partial)"); ax.set_xlabel(""); plt.show()

**Findings.** No duplicates; only volume has missing values (all early, in the training period); a few zero-volume days are genuine and kept; prices are internally consistent. Observations are not evenly spaced in calendar time because of weekends, holidays and one long closure.

### 2.1 Closing price and the chronological split

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4)); ax.plot(df["Date"], df["Close"], color="black", lw=1)
for name, g in df.groupby("split"):
    if name in SPLIT_COLORS: ax.axvspan(g["Date"].min(), g["Date"].max(), color=SPLIT_COLORS[name], alpha=0.10, label=f"{name} ({len(g)} rows)")
ax.set_title("Closing price (LKR) with the chronological train / validation / test split"); ax.set_ylabel("LKR"); ax.legend(loc="upper left"); plt.show()
display(df[df.split.isin(du.SPLITS)].groupby("split")["Close"].agg(["min", "mean", "max"]).loc[du.SPLITS].round(2))

### 2.2 Open, high, low and volume

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8), gridspec_kw={"width_ratios": [1.3, 1, 1.3]})
axes[0].fill_between(df["Date"], df["Low"], df["High"], color="#9ecae1", alpha=0.7, label="High-Low range"); axes[0].plot(df["Date"], df["Close"], color="black", lw=0.8, label="Close")
axes[0].set_title("Close and daily range"); axes[0].legend()
z = df.tail(120)
for c, col in [("Open", "#1f77b4"), ("High", "#2ca02c"), ("Low", "#d62728"), ("Close", "black")]: axes[1].plot(z["Date"], z[c], lw=1, color=col, label=c)
axes[1].set_title("Last 120 observations"); axes[1].legend(ncol=2, fontsize=7); axes[1].tick_params(axis="x", rotation=25)
axes[2].plot(df["Date"], df["Volume_raw"], color="#4c72b0", lw=0.6); axes[2].set_yscale("symlog", linthresh=100)
lo, hi = axes[2].get_ylim(); axes[2].vlines(df.loc[df["Volume_raw"].isna(), "Date"], lo, hi, color="red", alpha=0.5, lw=0.8, label="volume missing"); axes[2].set_ylim(lo, hi)
axes[2].set_title("Volume (symlog scale)"); axes[2].legend(); plt.show()

### 2.3 Daily returns, moving averages and correlations
We use **log returns** $r_t=\ln(P_t/P_{t-1})$.

In [ ]:
log_ret = np.log(df["Close"] / df["Close"].shift(1)); r = log_ret.dropna()
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8), gridspec_kw={"width_ratios": [1.5, 1, 1.2]})
axes[0].plot(df["Date"], log_ret * 100, lw=0.6, color="#4c72b0"); axes[0].set_title("Daily log return (%)")
axes[1].hist(r * 100, bins=100, color="#4c72b0"); axes[1].set_yscale("log"); axes[1].set_title("Distribution (log-count)")
n = 250; axes[2].plot(df["Date"].tail(n), df["Close"].tail(n), color="black", lw=1, label="Close")
for w, c in [(5, "#d62728"), (10, "#ff7f0e"), (20, "#1f77b4")]: axes[2].plot(df["Date"].tail(n), df["Close"].rolling(w).mean().tail(n), lw=1.1, color=c, label=f"MA{w}")
axes[2].set_title(f"Moving averages, last {n} obs."); axes[2].legend(fontsize=7); axes[2].tick_params(axis="x", rotation=25); plt.show()
print(f"std {r.std():.4f} | skew {stats.skew(r):.2f} | excess kurtosis {stats.kurtosis(r):.1f} | |return|>10%: {(r.abs() > 0.10).sum()} | unchanged-close days: {(df['Close'].diff() == 0).mean():.1%}")
top = r.abs().sort_values(ascending=False).head(6).index
display(pd.DataFrame({"date": df.loc[top, "Date"].dt.date, "close": df.loc[top, "Close"], "log_return": r[top]}).reset_index(drop=True).round(4))
basic = pd.DataFrame({"Open": df["Open"], "High": df["High"], "Low": df["Low"], "Close": df["Close"], "log(1+Vol)": np.log1p(df["Volume_raw"]), "Change %": df["Change_pct"], "log return": log_ret}).dropna()
corr = basic.corr(); fig, ax = plt.subplots(figsize=(6, 4.8)); im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right"); ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
[ax.text(j, i, f"{corr.iat[i, j]:.2f}", ha="center", va="center", fontsize=8) for i in range(len(corr)) for j in range(len(corr))]
ax.grid(False); fig.colorbar(im, fraction=0.046); ax.set_title("Correlation of the basic variables"); plt.show()
display(pd.DataFrame({"missing values": raw.isna().sum(), "share of rows": raw.isna().mean().round(4)}))

**Findings.** Returns are mostly tiny with occasional very large moves (heavy tails, volatility clusters). These are genuine and **kept**. Price levels are almost perfectly correlated with each other, while returns are nearly unrelated to anything: predicting a price *level* looks easy, predicting the *move* is hard.

## 3. Data preprocessing and leakage prevention
**Leakage** means the model gets information it would not have in real forecasting. We prevent it by: (1) a **chronological** split, never shuffled; (2) **backward-looking** features only (rows <= *t*); (3) the target (next return) is **never a feature**; (4) the missing-volume median is computed on **training rows only**; (5) **walk-forward** out-of-fold predictions for stacking (Section 7). The forecast is assumed to be made **after the close** of day *t*, so that day's OHLCV is legitimately known.

In [ ]:
volume_fill = float(df.loc[df["split"] == "train", "Volume_raw"].median())
print(f"Volume fill (TRAINING median): {volume_fill:,.0f} | median of all rows would be {df['Volume_raw'].median():,.0f} (not used: leaks)")
feats = pp.build_features(df, volume_fill); full = pp.assemble(df, feats); FEATURES = list(feats.columns)
has_f, has_t = full[FEATURES].notna().all(axis=1), full["target_log_return"].notna(); model_ready = full[has_f & has_t].copy()
display(pd.DataFrame({"rows": [len(full), int((~has_f).sum()), int((~has_t).sum()), len(model_ready)]},
    index=["records in CSV", "dropped: warm-up (lag/rolling windows unavailable)", "dropped: last row (no next observation)", "model-ready rows"]))
saved = pd.read_csv(PROCESSED_DIR / "daily_model_ready.csv", parse_dates=["date", "next_date"])
cols = FEATURES + ["close_t", "target_log_return", "target_price_next"]
print("Rebuilt features/targets identical to data/processed/daily_model_ready.csv:",
      bool(np.allclose(model_ready[cols].to_numpy(float), saved[cols].to_numpy(float), rtol=0, atol=1e-9) and (model_ready["date"].to_numpy() == saved["date"].to_numpy()).all()))
tbl = (full[full.split.isin(du.SPLITS)].groupby("split").agg(rows_before_drop=("date", "size"), first_date=("date", "min"), last_date=("date", "max")).loc[du.SPLITS])
tbl["rows_after_drop"] = model_ready.groupby("split").size().loc[tbl.index]; display(tbl)
groups = {"Price lags (ratio to today's close)": [f for f in FEATURES if "lag" in f and "ratio" in f], "Moving averages (close / MA)": [f for f in FEATURES if "over_ma" in f and "volume" not in f],
          "Returns": [f for f in FEATURES if f.startswith("log_ret")], "Volatility": [f for f in FEATURES if f.startswith("volatility")],
          "Intraday (ratio to close)": [f for f in FEATURES if f.endswith("over_close")], "Volume": ["log_volume", "volume_missing", "volume_over_ma20"],
          "Other": ["change_pct", "day_of_week", "month"]}
display(pd.Series({g: ", ".join(v) for g, v in groups.items()}, name="features").to_frame()); print(len(FEATURES), "features")

**Leakage test 1 (truncation).** For many origins *t* we rebuild the features from **only the data up to *t*** and check they equal the full-pipeline features. **Test 2:** validation/test rows must use earlier history across the split boundary, as a real forecaster would.

In [ ]:
rng = np.random.default_rng(0); fv = int(df.index[df.split == "validation"][0]); ft = int(df.index[df.split == "test"][0]); vi = model_ready.index.to_numpy()
origins = sorted(set(rng.choice(vi, 40, replace=False).tolist()) | {int(vi[0]), fv - 1, fv, ft - 1, ft})
bad = [t for t in origins if not np.allclose(pp.build_features(df.iloc[: t + 1].reset_index(drop=True), volume_fill).iloc[-1].to_numpy(float), feats.iloc[t].to_numpy(float), equal_nan=True)]
print(f"Truncation test on {len(origins)} origins incl. first validation/test rows: mismatches = {len(bad)}")
for name, t in [("first validation row", fv), ("first test row", ft)]:
    b = t - 20; print(f"{name} {df.loc[t, 'Date'].date()}: 20-observation lag comes from {df.loc[b, 'Date'].date()} (split: {df.loc[b, 'split']}), correct: {bool(np.isclose(feats.loc[t, 'close_lag20_ratio'] * df.loc[t, 'Close'], df.loc[b, 'Close']))}")
print("Target not among the features:", "target_log_return" not in FEATURES and "target_price_next" not in FEATURES)
print("close_t*exp(target) reproduces the next close; max diff:", f"{np.abs(model_ready['close_t'] * np.exp(model_ready['target_log_return']) - model_ready['target_price_next']).max():.1e} LKR")

**The target in the three periods.** The next-day return is much more volatile in the test period, and the 'rise today -> fall tomorrow' relation weakens. Models learned on calm data are tested on turbulent data.

In [ ]:
mr = model_ready.reset_index(drop=True)
tstat = mr.groupby("split")["target_log_return"].agg(["count", "mean", "std", "min", "max"]).loc[du.SPLITS]
tstat["corr(next return, today's return)"] = [np.corrcoef(mr.loc[mr.split == s, "target_log_return"], mr.loc[mr.split == s, "log_ret_lag0"])[0, 1] for s in tstat.index]
display(tstat.round(4))
fig, ax = plt.subplots(figsize=(10, 3.2))
for s in du.SPLITS: ax.hist(mr.loc[mr.split == s, "target_log_return"], bins=120, alpha=0.55, density=True, color=SPLIT_COLORS[s], label=s)
ax.set_xlim(-0.2, 0.2); ax.set_title("Target (next-observation log return) by period"); ax.legend(); plt.show()

## 4. Persistence baseline
Everything from here on uses the **saved** models and results. The next cell loads them, predicts the validation and test observations, converts predicted log returns to prices ($\widehat P_{t+1}=P_te^{\widehat y_t}$), recomputes all metrics and cross-checks them against the saved `metrics_all.csv`. No model is retrained.

In [ ]:
df_mr, FEATS = du.load_model_ready(); assert FEATS == FEATURES
X_ret = df_mr[FEATS]; X_raw = du.raw_price_feature_table(df_mr)
MASK = {s: (df_mr["split"] == s).to_numpy() for s in du.SPLITS}
metrics_saved = pd.read_csv(RESULTS_DIR / "metrics_all.csv"); summary = json.loads((RESULTS_DIR / "summary.json").read_text())
MODELS = {(a, k): joblib.load(MODELS_DIR / registry["registry"][f"{a}_{k}"]) for a in ["return", "rawprice"] for k in ["random_forest", "xgboost", "stacking"]}
print(f"Loaded {len(MODELS)} saved models (nothing is retrained).")
PRICE, LOGRET = {}, {}
for a in ["return", "rawprice"]:
    X = X_ret if a == "return" else X_raw
    for k in ["baseline", "random_forest", "xgboost", "stacking"]:
        for s in ["validation", "test"]:
            m = MASK[s]; close = df_mr.loc[m, "close_t"].to_numpy()
            if k == "baseline":
                PRICE[(a, k, s)] = close.copy()
                if a == "return": LOGRET[(k, s)] = np.zeros(len(close))
                continue
            out = MODELS[(a, k)].predict(X[m])
            if a == "return": LOGRET[(k, s)] = out; PRICE[(a, k, s)] = ev.reconstruct_price(close, out)
            else: PRICE[(a, k, s)] = out
live = pd.DataFrame([{"approach": a, "model": LABELS[k], "split": s, **ev.evaluate_prices(df_mr.loc[MASK[s], "close_t"].to_numpy(), df_mr.loc[MASK[s], "target_price_next"].to_numpy(), PRICE[(a, k, s)])}
                     for a in ["return", "rawprice"] for k in ["baseline", "random_forest", "xgboost", "stacking"] for s in ["validation", "test"]])
chk = live.merge(metrics_saved, on=["approach", "model", "split"], suffixes=("_l", "_s"))
print("Max difference, recomputed metrics vs saved metrics_all.csv:", f"{max(np.nanmax(np.abs(chk[c + '_l'] - chk[c + '_s'])) for c in ['MAE', 'RMSE', 'R2', 'DirAcc_incl_unchanged', 'DirAcc_excl_unchanged', 'Return_R2']):.2e}")
TRAIN_CLOSE_MAX = float(df_mr.loc[MASK["train"], "close_t"].max())
def metrics_table(approach="return", keys=("baseline", "random_forest", "xgboost", "stacking"), splits=("validation", "test")):
    t = live[(live.approach == approach) & live.model.isin([LABELS[k] for k in keys]) & live.split.isin(splits)].set_index(["split", "model"])
    return t[["MAE", "RMSE", "R2", "DirAcc_incl_unchanged", "DirAcc_excl_unchanged", "Return_R2"]].loc[[(s, LABELS[k]) for s in splits for k in keys]].round(4)

The **persistence** forecast says *tomorrow's close = today's close* (a predicted log return of 0). It is a hard benchmark because prices change little between observations, and **a model is only useful if it beats it**. Its MAE is the mean absolute price change; it never predicts a direction, so directional accuracy *excluding* unchanged days is undefined.

In [ ]:
display(metrics_table("return", keys=("baseline",)))
for s in ["validation", "test"]:
    c, a = df_mr.loc[MASK[s], "close_t"].to_numpy(), df_mr.loc[MASK[s], "target_price_next"].to_numpy()
    print(f"{s:10s}: unchanged-close days {(a == c).mean():.1%} | mean absolute price change {np.abs(a - c).mean():.4f} LKR")

## 5. Random Forest (Bagging)
**How it works.** (1) *Bootstrap sampling:* each tree trains on a random sample of rows drawn with replacement. (2) *Many decision trees:* each asks simple questions about the features, using only a random subset of features at each split. (3) *Averaging:* the forest predicts the **average** of its trees, which cancels noise (lower variance).

| Parameter | Meaning |
|---|---|
| `n_estimators` | number of trees |
| `max_depth` | how many questions in a row a tree may ask |
| `min_samples_leaf` | minimum rows per leaf; larger = smoother, less over-fitting |
| `max_features` | share of features tried at each split |

In [ ]:
rf = MODELS[("return", "random_forest")]; hp_rf = registry["hyperparameters"]["return"]["random_forest"]
display(pd.DataFrame({"selected value": hp_rf}))
tune_rf = pd.read_csv(RESULTS_DIR / "tuning_return_random_forest.csv"); display(tune_rf.head(6).round(5))
grown = max(t.get_depth() for t in rf.estimators_); ties = tune_rf[np.isclose(tune_rf["val_RMSE"], tune_rf["val_RMSE"].min(), atol=1e-12)]
print(f"max_depth setting {rf.max_depth}; deepest tree actually grown {grown}; {len(ties)} candidates tie for best validation RMSE (max_depth values {sorted(ties['max_depth'].tolist())}).")
print("With large min_samples_leaf the depth limit never binds, so tied values give the same fitted model; which is reported as 'selected' is arbitrary.")

**Optional demonstration** (in memory only; nothing is saved). We fit a forest with the same settings and compare it with the saved one. Set `DEMO_REFIT = False` to skip.

In [ ]:
DEMO_REFIT = True
if DEMO_REFIT:
    tr, va = MASK["train"], MASK["validation"]; t0 = time.time()
    rf_demo = em.make_random_forest(hp_rf).fit(X_ret[tr], df_mr.loc[tr, "target_log_return"].to_numpy())
    print(f"Fitted in {time.time() - t0:.1f}s; max difference to saved model (validation log returns): {np.abs(rf_demo.predict(X_ret[va]) - rf.predict(X_ret[va])).max():.2e}")
display(metrics_table("return", keys=("baseline", "random_forest")))
fi = pd.read_csv(RESULTS_DIR / "feature_importance.csv")
def importance(approach, label, kind="built-in", cols=None):
    s = fi[(fi.approach == approach) & (fi.model == label) & fi.importance_type.str.startswith(kind)].set_index("feature")["importance"]
    return s.loc[cols] if cols is not None else s
rf_imp = importance("return", LABELS["random_forest"], "built-in", FEATS)
print("Saved importances equal the saved model's feature_importances_:", bool(np.allclose(rf_imp.to_numpy(), rf.feature_importances_)))
top = rf_imp.sort_values().tail(12); fig, ax = plt.subplots(figsize=(7, 3.8)); ax.barh(top.index, top.values, color="#2ca02c"); ax.set_title("Random Forest: top 12 features"); plt.show()

## 6. XGBoost (Boosting)
**How it works.** Trees are built **one after another**. The first makes a rough prediction; each next tree is trained on the **errors (residuals)** left by the previous ones, adding a small correction scaled by the **learning rate**. The total error falls round by round. XGBoost is a fast, regularised implementation.

| Parameter | Meaning |
|---|---|
| `n_estimators` | number of boosting rounds |
| `learning_rate` | size of each correction step (small = careful, needs more rounds) |
| `max_depth` | tree size (small trees = weak learners) |
| `min_child_weight` | minimum data a leaf needs (larger = more cautious) |
| `subsample`, `colsample_bytree`, `reg_lambda` | row/feature sampling (0.8) and shrinkage penalty (1), fixed |

In [ ]:
xgb = MODELS[("return", "xgboost")]; hp_xgb = registry["hyperparameters"]["return"]["xgboost"]
display(pd.DataFrame({"selected value": hp_xgb}))
tune_xgb = pd.read_csv(RESULTS_DIR / "tuning_return_xgboost.csv"); display(tune_xgb.head(6).round(5))
top10 = tune_xgb.head(10)["val_RMSE"]
print(f"The ten best XGBoost candidates differ by only {(top10.max() / top10.min() - 1) * 100:.2f}% in validation RMSE, so the winner can change slightly between computers or library versions.")
Xtr, ytr = X_ret[MASK["train"]], df_mr.loc[MASK["train"], "target_log_return"].to_numpy(); Xva, yva = X_ret[MASK["validation"]], df_mr.loc[MASK["validation"], "target_log_return"].to_numpy()
rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2))); ks = list(range(10, hp_xgb["n_estimators"] + 1, 10))
trc = [rmse(ytr, xgb.predict(Xtr, iteration_range=(0, k))) for k in ks]; vac = [rmse(yva, xgb.predict(Xva, iteration_range=(0, k))) for k in ks]
fig, ax = plt.subplots(figsize=(8, 3.8)); ax.plot(ks, trc, label="training rows", color="#1f77b4"); ax.plot(ks, vac, label="validation rows", color="#d62728")
ax.axhline(rmse(yva, 0 * yva), color="grey", ls="--", lw=1, label="validation: always predict 0"); ax.axvline(hp_xgb["n_estimators"], color="black", ls=":", label="selected rounds")
ax.set_xlabel("boosting rounds"); ax.set_ylabel("RMSE of log return"); ax.set_title("Error after k boosting rounds (saved model, no retraining)"); ax.legend(); plt.show()
if DEMO_REFIT:
    t0 = time.time(); xd = em.make_xgboost(hp_xgb).fit(Xtr, ytr)
    print(f"In-memory refit in {time.time() - t0:.1f}s; max difference to saved model on validation: {np.abs(xd.predict(Xva) - xgb.predict(Xva)).max():.2e}")
display(metrics_table("return", keys=("baseline", "xgboost")))
xgb_imp = importance("return", LABELS["xgboost"], "built-in", FEATS)
print("Saved importances equal the saved model's:", bool(np.allclose(xgb_imp.to_numpy(), xgb.feature_importances_)))
top = xgb_imp.sort_values().tail(12); fig, ax = plt.subplots(figsize=(7, 3.8)); ax.barh(top.index, top.values, color="#d62728"); ax.set_title("XGBoost: top 12 features"); plt.show()

**Random Forest, XGBoost, LightGBM and CatBoost (conceptual; only the first two were run).**

| | Random Forest | XGBoost | LightGBM | CatBoost |
|---|---|---|---|---|
| Type | Bagging | Boosting | Boosting | Boosting |
| Tree growth | independent trees on bootstrap samples | level-wise, depth-limited | leaf-wise, histogram based | symmetric ("oblivious") trees |
| Main idea | averaging reduces variance | regularised objective, shrinkage | very fast on large data | ordered boosting, strong categorical support |
| Used here | yes | yes | no | no |

## 7. Stacking ensemble (Random Forest + XGBoost -> Ridge)
A **Ridge** regression (linear regression with a weight penalty) learns how to combine the two base learners. Ordinary K-fold cross-validation would let a base model train on **future** rows to predict **past** rows, which leaks. Instead we use an **expanding window** (walk-forward): train on the first block, predict the next; train on everything so far, predict the following block, and so on. Each out-of-fold prediction therefore comes from a model that saw **only earlier rows**; Ridge learns from those only. Afterwards the base learners are re-fitted on all training rows.

In [ ]:
st = MODELS[("return", "stacking")]; folds = pd.DataFrame(st.fold_info_); display(folds)
starts = folds["predict_rows"].str.split("..", regex=False).str[0].astype(int)
print("Every fold trains only on rows before the block it predicts:", bool((folds["n_train"] <= starts).all()),
      "| training windows end before predicted blocks (dates):", bool((pd.to_datetime(folds["train_end_date"]) < pd.to_datetime(folds["predict_start_date"])).all()))
fig, ax = plt.subplots(figsize=(10, 3))
for i, f in folds.iterrows():
    s = int(f["predict_rows"].split("..")[0]); ax.barh(i, f["n_train"], color="#9ecae1", label="training window" if i == 0 else None); ax.barh(i, f["n_predict"], left=s, color="#fd8d3c", label="predicted block (out-of-fold)" if i == 0 else None)
ax.invert_yaxis(); ax.set_yticks(range(len(folds))); ax.set_yticklabels([f"fold {k}" for k in folds["fold"]]); ax.set_xlabel("row within training period (old -> new)"); ax.set_title("Expanding-window folds"); ax.legend(loc="lower left"); plt.show()
oof, y_oof = st.oof_predictions_, st.oof_target_; ok = ~np.isnan(oof).any(axis=1)
print(f"Out-of-fold rows used by the meta-model: {int(ok.sum())} of {len(oof)} (the first {int((~ok).sum())} rows are only fold 1's starting window)")
display(pd.DataFrame({"RF OOF": oof[ok, 0], "XGB OOF": oof[ok, 1], "actual next log return": y_oof[ok]}).corr().round(3))
display(st.meta_weights().round(4)); print("Ridge alpha chosen on validation:", st.ridge_alpha); display(pd.read_csv(RESULTS_DIR / "tuning_return_stacking_ridge_alpha.csv").round(5))
display(metrics_table("return"))

## 8. Return-based versus raw-price experiment
| | Primary: **return-based** | Secondary: **raw-price** |
|---|---|---|
| Model predicts | next log return | the next closing price directly |
| Price forecast | $P_t e^{\widehat y_t}$ | model output |
| Features | scale-free ratios | price levels |

Tree models average values seen in training, so they **cannot predict above the largest training price**. If the price later rises far above that, a raw-price tree model must fall behind.

In [ ]:
hp_raw = registry["hyperparameters"]["rawprice"]; display(pd.DataFrame({"Random Forest": hp_raw["random_forest"], "XGBoost": hp_raw["xgboost"]}))
print("Ridge alpha (raw-price stack):", hp_raw["ridge_alpha"], "| raw-price features:", X_raw.shape[1]); display(metrics_table("rawprice"))
keys = ["baseline", "random_forest", "xgboost", "stacking"]; tm = MASK["test"]
actual_t = df_mr.loc[tm, "target_price_next"].to_numpy(); close_t = df_mr.loc[tm, "close_t"].to_numpy(); d_t = df_mr.loc[tm, "next_date"]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.3), gridspec_kw={"width_ratios": [2, 1.1]})
axes[0].plot(d_t, actual_t, color="black", lw=1.6, label="Actual next close")
for k in keys[1:]: axes[0].plot(d_t, PRICE[("rawprice", k, "test")], lw=1, color=COLORS[LABELS[k]], label=f"{LABELS[k]} (raw-price)")
axes[0].axhline(TRAIN_CLOSE_MAX, color="black", ls="--", lw=0.9, label=f"highest training close ({TRAIN_CLOSE_MAX:.1f})"); axes[0].set_title("Raw-price experiment, test period"); axes[0].legend(fontsize=8, loc="upper left"); axes[0].tick_params(axis="x", rotation=25)
names = [LABELS[k] for k in keys[1:]]; x = np.arange(3); w = 0.38
g = lambda a, n: live[(live.approach == a) & (live.split == "test") & (live.model == n)].RMSE.iloc[0]
axes[1].bar(x - w / 2, [g("return", n) for n in names], w, label="return-based", color="#1f77b4"); axes[1].bar(x + w / 2, [g("rawprice", n) for n in names], w, label="raw-price", color="#ff7f0e")
axes[1].axhline(g("return", LABELS["baseline"]), color="grey", ls="--", label="persistence"); axes[1].set_xticks(x); axes[1].set_xticklabels(["RF", "XGBoost", "Stacking"]); axes[1].set_title("Test RMSE (LKR)"); axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()
above = close_t > TRAIN_CLOSE_MAX
print(f"Highest training close {TRAIN_CLOSE_MAX:.2f}; highest test next close {actual_t.max():.2f}; highest raw-price prediction on test: " + str({LABELS[k]: round(float(PRICE[('rawprice', k, 'test')].max()), 2) for k in keys[1:]}))
rows = {}
for a in ["return", "rawprice"]:
    for k in keys:
        if k == "baseline" and a == "rawprice": continue
        p = PRICE[(a, k, "test")]; nm = LABELS[k] + ("" if k == "baseline" else f" [{'return-based' if a == 'return' else 'raw-price'}]")
        rows[nm] = {f"close above training max (n={int(above.sum())})": np.abs(actual_t - p)[above].mean() if above.any() else np.nan, f"within training range (n={int((~above).sum())})": np.abs(actual_t - p)[~above].mean()}
display(pd.DataFrame(rows).T.round(3))

**Conclusion.** Predicting returns and converting back is the right design here: the raw-price models are tied to the training price range. The return-based models must still be judged against the baseline (Section 9).

## 9. Model evaluation and comparison (return-based models, prices in LKR, same observations for every model)
**MAE / RMSE:** average price error in LKR (RMSE punishes large errors more); lower is better. **R²:** always near 1 here because today's price already explains tomorrow's, so it barely separates models. **Return R²:** the same idea on the price *move*; near or below 0 means no better than "no change". **Directional accuracy:** share of correct up/down calls, including and excluding unchanged days. **MAE improvement** has a 95% moving-block bootstrap interval; an interval containing 0 means the difference is not reliable.

In [ ]:
def comparison_table(split):
    t = live[(live.approach == "return") & (live.split == split)].set_index("model"); s = metrics_saved[(metrics_saved.approach == "return") & (metrics_saved.split == split)].set_index("model"); b = t.loc[LABELS["baseline"]]
    o = t[["MAE", "RMSE", "R2", "Return_R2", "DirAcc_incl_unchanged", "DirAcc_excl_unchanged"]].copy()
    o["RMSE improvement vs baseline (%)"] = (1 - t["RMSE"] / b["RMSE"]) * 100; o["MAE improvement vs baseline (%)"] = s["MAE_gain_vs_baseline_pct"]; o["95% CI low"] = s["MAE_gain_CI95_low"]; o["95% CI high"] = s["MAE_gain_CI95_high"]
    return o.loc[[LABELS[k] for k in keys]].round(4)
for s in ["validation", "test"]: print(f"--- {s.upper()} ({int(MASK[s].sum())} observations)"); display(comparison_table(s))
fig, axes = plt.subplots(1, 3, figsize=(15, 4.1))
for ax, col, ttl in [(axes[0], "MAE", "MAE (LKR), lower is better"), (axes[1], "RMSE", "RMSE (LKR), lower is better"), (axes[2], "DirAcc_excl_unchanged", "Directional accuracy (days price moved)")]:
    for i, (s, h, al) in enumerate([("validation", "", 0.55), ("test", "//", 0.95)]):
        v = [live[(live.approach == "return") & (live.split == s) & (live.model == LABELS[k])][col].iloc[0] for k in keys]
        bars = ax.bar(np.arange(4) + (i - 0.5) * 0.38, v, 0.38, color=[COLORS[LABELS[k]] for k in keys], hatch=h, edgecolor="white", alpha=al)
        [ax.text(b.get_x() + b.get_width() / 2, y, f"{y:.3f}" if col != "DirAcc_excl_unchanged" else f"{y:.1%}", ha="center", va="bottom", fontsize=7) for b, y in zip(bars, v) if not np.isnan(y)]
    ax.set_xticks(range(4)); ax.set_xticklabels(["Baseline", "RF", "XGBoost", "Stacking"]); ax.set_title(ttl)
axes[2].axhline(0.5, color="black", ls="--", lw=0.7); axes[2].text(0, 0.02, "n/a", ha="center", fontsize=8)
axes[0].bar([], [], color="grey", alpha=0.55, label="validation (light)"); axes[0].bar([], [], color="grey", hatch="//", label="test (hatched)"); axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

**Is directional accuracy better than chance?** On days the price moved, a coin flip is right about 50% of the time. The p-value is an exact binomial test (assumes independent days; six comparisons, so a rough guide).

In [ ]:
def direction_table(split):
    m = MASK[split]; c = df_mr.loc[m, "close_t"].to_numpy(); a = df_mr.loc[m, "target_price_next"].to_numpy(); act = np.sign(a - c); mv = act != 0; rows = []
    for k in keys[1:]:
        pdir = np.sign(PRICE[("return", k, split)] - c); h, n = int((pdir[mv] == act[mv]).sum()), int(mv.sum())
        rows.append({"model": LABELS[k], "days price moved": n, "correct direction": h, "accuracy": h / n, "p-value (vs 50%)": stats.binomtest(h, n, 0.5).pvalue})
    return pd.DataFrame(rows).set_index("model").round(4), float((act[mv] > 0).mean()), float((act == 0).mean())
for s in ["validation", "test"]:
    t_, up, fl = direction_table(s); print(f"--- {s}: {up:.1%} of moves were increases; {fl:.1%} of days unchanged"); display(t_)

### Actual versus predicted closing price, and errors over time
Forecasts hug the actual price because tomorrow is close to today; in the zoom they look like the actual series **shifted by one day**, the signature of a next-day forecast driven by today's price.

In [ ]:
def plot_avp(split, zoom=60):
    d = df_mr.loc[MASK[split], "next_date"]; act = df_mr.loc[MASK[split], "target_price_next"]
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={"width_ratios": [2, 1]})
    for ax, sl, ttl in [(axes[0], slice(None), f"{split.capitalize()}: actual vs predicted next close"), (axes[1], slice(-zoom, None), f"Zoom: last {zoom}")]:
        ax.plot(d.iloc[sl], act.iloc[sl], color="black", lw=1.6, label="Actual")
        [ax.plot(d.iloc[sl], PRICE[("return", k, split)][sl], lw=1, alpha=0.85, color=COLORS[LABELS[k]], label=LABELS[k]) for k in keys]
        ax.set_title(ttl); ax.set_ylabel("LKR"); ax.tick_params(axis="x", rotation=25)
    axes[0].legend(fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()
plot_avp("test"); plot_avp("validation"); print("Straight segments in the validation plot (spring 2020) are the long market closure: no observations exist there.")
fig, axes = plt.subplots(4, 1, figsize=(12, 7.5), sharex=True, sharey=True)
for ax, k in zip(axes, keys):
    e = actual_t - PRICE[("return", k, "test")]; ax.plot(d_t, e, lw=0.9, color=COLORS[LABELS[k]]); ax.axhline(0, color="black", lw=0.7); ax.set_title(f"{LABELS[k]} (MAE {np.abs(e).mean():.3f})", fontsize=9, loc="left"); ax.set_ylabel("error (LKR)")
axes[-1].set_xlabel("date predicted"); plt.suptitle("Prediction error over time, test set (actual - predicted)", y=1.0); plt.tight_layout(); plt.show()

### Residual analysis (test set)
Residuals = actual - predicted. A good model's residuals centre on 0, show no pattern against the prediction, and are not related from day to day (**autocorrelation**).

In [ ]:
res = {k: actual_t - PRICE[("return", k, "test")] for k in keys}
res_tbl = pd.DataFrame({LABELS[k]: ev.residual_summary(actual_t, PRICE[("return", k, "test")]) for k in keys}).T
sr = pd.read_csv(RESULTS_DIR / "residual_summary.csv").query("approach == 'return' and split == 'test'").set_index("model").loc[res_tbl.index, res_tbl.columns]
print("Residual statistics vs saved residual_summary.csv, max difference:", f"{np.abs(res_tbl.to_numpy() - sr.to_numpy()).max():.2e}"); display(res_tbl.round(3))
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for j, k in enumerate(keys):
    p = PRICE[("return", k, "test")]; axes[0, j].hist(res[k], bins=40, color=COLORS[LABELS[k]], alpha=0.85); axes[0, j].axvline(0, color="black", lw=0.8); axes[0, j].set_title(LABELS[k], fontsize=9); axes[0, j].set_xlabel("residual (LKR)")
    axes[1, j].scatter(p, res[k], s=6, alpha=0.6, color=COLORS[LABELS[k]]); axes[1, j].axhline(0, color="black", lw=0.8); axes[1, j].set_xlabel("predicted price (LKR)")
plt.suptitle("Residual distribution (top) and residual vs predicted price (bottom)", y=1.0); plt.tight_layout(); plt.show()
def acf(x, n=15): x = np.asarray(x) - np.mean(x); return np.array([(x[:-i] * x[i:]).sum() / (x ** 2).sum() for i in range(1, n + 1)])
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for k in keys: axes[0].plot(range(1, 16), acf(res[k]), marker="o", ms=3, lw=1, color=COLORS[LABELS[k]], label=LABELS[k])
band = 1.96 / np.sqrt(len(actual_t)); axes[0].axhspan(-band, band, color="grey", alpha=0.2, label="approx. 95% band"); axes[0].axhline(0, color="black", lw=0.7); axes[0].set_xlabel("lag"); axes[0].set_title("Residual autocorrelation"); axes[0].legend(fontsize=7)
for k in keys:
    (osm, osr), (sl, ic, _) = stats.probplot(res[k], dist="norm"); axes[1].plot(osm, osr, ".", ms=3, color=COLORS[LABELS[k]], label=LABELS[k])
axes[1].plot(osm, sl * osm + ic, color="black", lw=0.8); axes[1].set_title("Q-Q plot (curved ends = heavy tails)"); axes[1].legend(fontsize=7); plt.tight_layout(); plt.show()

### Do the forecasts capture the move? Feature importance. Regimes.
If a model captured the move, dots would lie along an upward line; notice how small the predicted returns are compared with actual ones. Permutation importance (validation set only) is tiny and noisy.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6.5), sharey=True)
for i, s in enumerate(["validation", "test"]):
    y = df_mr.loc[MASK[s], "target_log_return"].to_numpy()
    for j, k in enumerate(keys[1:]):
        p = LOGRET[(k, s)]; axes[i, j].scatter(p, y, s=7, alpha=0.5, color=COLORS[LABELS[k]]); axes[i, j].axhline(0, color="black", lw=0.6); axes[i, j].axvline(0, color="black", lw=0.6); axes[i, j].set_ylim(-0.25, 0.25)
        axes[i, j].set_title(f"{LABELS[k]} | {s}\ncorr {np.corrcoef(p, y)[0, 1]:+.3f}, pred std {p.std():.4f} vs actual {y.std():.4f}", fontsize=8); axes[i, j].set_xlabel("predicted log return")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for i, (kind, ttl) in enumerate([("built-in", "built-in importance"), ("permutation", "permutation importance (validation)")]):
    for j, k in enumerate(["random_forest", "xgboost"]):
        s = importance("return", LABELS[k], kind).sort_values().tail(12); axes[i, j].barh(s.index, s.values, color=COLORS[LABELS[k]]); axes[i, j].set_title(f"{LABELS[k]}: {ttl}", fontsize=9)
plt.suptitle("Feature importance, return-based models (top 12)", y=1.0); plt.tight_layout(); plt.show()
for k in ["random_forest", "xgboost"]: pi = importance("return", LABELS[k], "permutation"); print(f"{LABELS[k]}: {int((pi < 0).sum())} of {len(pi)} features have negative permutation importance")
display(st.meta_weights()[["base_learner", "weight_on_prediction"]].round(4))
y_t = df_mr.loc[tm, "target_log_return"].to_numpy(); rows = []
for b in np.array_split(np.arange(len(actual_t)), 4):
    row = {"period": f"{d_t.iloc[b[0]].date()} to {d_t.iloc[b[-1]].date()}", "actual volatility (std of log return)": y_t[b].std()}
    for k in keys: row[LABELS[k] + " MAE"] = np.abs(actual_t[b] - PRICE[("return", k, "test")][b]).mean()
    rows.append(row)
display(pd.DataFrame(rows).set_index("period").round(4))

## 10. Interpretation of the results
The text below is **generated from the numbers above** each time the notebook runs, so it always describes the files actually loaded.

In [ ]:
val = live[(live.approach == "return") & (live.split == "validation")].set_index("model"); tst = live[(live.approach == "return") & (live.split == "test")].set_index("model")
B = LABELS["baseline"]; ENS = [LABELS[k] for k in keys[1:]]; short = lambda m: m.split(" (")[0]; best_val, best_test = val.RMSE.idxmin(), tst.RMSE.idxmin(); L = []
L.append((f"**Validation.** The lowest RMSE belongs to **{best_val}** ({val.loc[best_val, 'RMSE']:.4f} LKR vs {val.loc[B, 'RMSE']:.4f} for persistence, {(1 - val.loc[best_val, 'RMSE'] / val.loc[B, 'RMSE']) * 100:.1f}% lower). MAE is lower than the baseline by "
          + ", ".join(f"{short(m)} {(1 - val.loc[m, 'MAE'] / val.loc[B, 'MAE']) * 100:.1f}%" for m in ENS) + ".") if best_val != B else f"**Validation.** No ensemble beat persistence (RMSE {val.loc[B, 'RMSE']:.4f}).")
L.append((f"**Test.** Persistence has the lowest RMSE ({tst.loc[B, 'RMSE']:.4f}) and MAE ({tst.loc[B, 'MAE']:.4f}). Every ensemble is slightly *worse*: MAE higher by " + ", ".join(f"{short(m)} {(tst.loc[m, 'MAE'] / tst.loc[B, 'MAE'] - 1) * 100:.1f}%" for m in ENS) + ".")
         if best_test == B else f"**Test.** The lowest test RMSE belongs to **{best_test}** ({tst.loc[best_test, 'RMSE']:.4f} vs {tst.loc[B, 'RMSE']:.4f} for persistence).")
sv = metrics_saved[(metrics_saved.approach == "return") & (metrics_saved.split == "test")].set_index("model"); vd = []
for m in ENS:
    g, lo, hi = sv.loc[m, ["MAE_gain_vs_baseline_pct", "MAE_gain_CI95_low", "MAE_gain_CI95_high"]]
    vd.append(f"{short(m)}: {g:+.1f}% (95% interval {lo:+.1f}% to {hi:+.1f}%) -> " + ("reliably better" if lo > 0 else "reliably worse" if hi < 0 else "not reliably different from the baseline"))
L.append("**Is the test difference reliable?** MAE improvement over baseline on test: " + "; ".join(vd) + ".")
dt, up, flat = direction_table("test")
L.append("**Direction.** On test days the price moved, the models called the direction correctly " + ", ".join(f"{short(m)} {dt.loc[m, 'accuracy']:.1%} (p = {dt.loc[m, 'p-value (vs 50%)']:.3f})" for m in ENS) + ". A coin flip gives about 50%, so the forecasts carry some directional information, but they badly miss the *size* of moves, which dominates MAE and RMSE.")
cc = {s: np.corrcoef(mr.loc[mr.split == s, "target_log_return"], mr.loc[mr.split == s, "log_ret_lag0"])[0, 1] for s in du.SPLITS}; sd = {s: mr.loc[mr.split == s, "target_log_return"].std() for s in du.SPLITS}
vs = (mr.loc[mr.split == "test", "volatility_20"] > mr.loc[mr.split == "train", "volatility_20"].max()).mean()
L.append(f"**Why the test period is harder.** (a) The correlation between today's return and tomorrow's is {cc['train']:+.2f} (train), {cc['validation']:+.2f} (validation), {cc['test']:+.2f} (test), so a pattern like 'a rise is followed by a fall' fades. (b) The std of the next-day return is {sd['train']:.3f}, {sd['validation']:.3f}, {sd['test']:.3f}, and {vs:.0%} of test days have 20-day volatility above anything seen in training. These are observations; they do not prove the cause.")
ps = {m: LOGRET[(k, "test")].std() for m, k in zip(ENS, keys[1:])}
L.append("**Forecasts are cautious.** Std of the predicted test log return: " + ", ".join(f"{short(m)} {v:.4f}" for m, v in ps.items()) + f" vs {y_t.std():.4f} actual: the models make small adjustments around today's price and cannot anticipate big jumps.")
rw = live[(live.approach == "rawprice") & (live.split == "test")].set_index("model")
L.append("**Return-based vs raw-price.** Raw-price test RMSE: " + ", ".join(f"{short(m)} {rw.loc[m, 'RMSE']:.2f}" for m in ENS) + f" vs {tst.loc[B, 'RMSE']:.2f} for persistence. Their highest prediction was {max(PRICE[('rawprice', k, 'test')].max() for k in keys[1:]):.1f} LKR while training prices never exceeded {TRAIN_CLOSE_MAX:.1f} and test prices reach {actual_t.max():.1f}.")
L.append(f"**How firmly were hyper-parameters selected?** {len(ties)} Random Forest candidates tie exactly (max_depth {sorted(ties['max_depth'].tolist())}) and are the same fitted model; the ten best XGBoost candidates are within {(top10.max() / top10.min() - 1) * 100:.2f}% of each other. Different selected settings on different computers are therefore not alarming.")
display(Markdown("\n\n".join(L)))

**What the three ideas teach us.** Bagging averages noisy trees (with large leaves it is smooth and cautious). Boosting keeps fitting remaining errors, so with a noisy target it needs shallow trees, a small learning rate and limited rounds (Section 6). Stacking can only be as good as its base learners, whose predictions are similar. None can create information that is not in the features: the next-day move of this stock is largely unpredictable from past prices and volumes.

## 11. Limitations and final conclusions
### Limitations
1. **One stock, one period.** The 2021-2023 test period is a single unusually volatile regime.
2. **Not re-fitted.** Models were fitted on training only, not on training + validation; by the test period they are years old.
3. **Regime change.** The relationship learned in training weakened over time (Section 10).
4. **Small hyper-parameter search**, with several tied or near-tied candidates; XGBoost results can differ slightly between computers and library versions.
5. **Permutation importance is tiny and noisy**; importance is descriptive, not causal.
6. **Directional p-values** assume independent days and are uncorrected for multiple comparisons.
7. **No trading costs or strategy.** Predicting a price is not the same as profiting from it. Not financial advice.
8. **Interpretations are hypotheses**, supported by the numbers but not proven.

### Optional hybrid: Random Forest + ARIMA (not implemented)
It was **deliberately omitted**: a leakage-safe version needs ARIMA fitted on an expanding window of historical residuals and re-fitted at every origin, which is slow, fragile and hard to verify, and `statsmodels` is not in this project's environment. It is **future work**, together with re-fitting on train + validation, LightGBM/CatBoost, probabilistic forecasts and a leakage-safe hybrid.

In [ ]:
A = [f"**Q1. Can ensembles beat persistence?** Validation: {'yes, all three' if (val.loc[ENS, 'RMSE'] < val.loc[B, 'RMSE']).all() else 'not all'} (best: {best_val}). Test: {'no, the baseline was best' if best_test == B else 'best was ' + best_test}; differences were {'not reliable' if all(sv.loc[m, 'MAE_gain_CI95_low'] <= 0 <= sv.loc[m, 'MAE_gain_CI95_high'] for m in ENS) else 'reliable for at least one model'} per the bootstrap interval.",
     f"**Q2. Bagging, boosting or stacking?** Ranking by RMSE (best first). Validation: {' < '.join(short(m) for m in val.RMSE.sort_values().index)}. Test: {' < '.join(short(m) for m in tst.RMSE.sort_values().index)}. Differences among the ensembles are small.",
     f"**Q3. Does validation performance carry over?** {'No.' if (best_val != B and best_test == B) else 'Partly.'} The best validation model ({best_val}) was {'also best' if best_val == best_test else 'not best'} on test; validation scores were optimistic because settings were tuned there and the test market behaved differently.",
     "**Main conclusion.** Next-day closing prices are dominated by today's price. Predicting the log return and converting back is the right design (raw-price trees cannot extrapolate). The ensembles extract a small amount of signal on past data, but it is too weak and unstable to reliably beat persistence on later data. Reporting this honestly is the point of the exercise."]
display(Markdown("\n\n".join(A)))

## 12. AI usage disclosure
**Tool.** Claude (Anthropic) was used as a coding and writing assistant through a chat interface during this project. *(Add the model name/version and dates here if your institution requires them.)*

**Student's role.** I (the student) defined the assignment, chose the dataset and the research question, and made the methodological decisions: forecasting the next trading observation, a chronological 70/15/15 split with no shuffling, a leakage-safe design, comparing bagging, boosting and stacking against a persistence baseline, and using walk-forward validation for stacking. Claude gave suggestions and options, such as predicting the log return instead of the raw price. I reviewed each suggestion and approved it before it was implemented. I approved each phase before the next one began.

**What the AI did.** Claude wrote the Python code in `src/`, the Jupyter notebook and the Streamlit dashboard, wrote the verification scripts, and drafted the explanatory text in the notebook and the documentation. I reviewed this text.

**What I ran myself.** I ran preprocessing, model training and verification on my own MacBook Air M1 (Python 3.11). The models and results used by the notebook and dashboard come from that run. When I noticed that the hyperparameters selected on my Mac differed from the AI's reference run, I asked for the cause to be investigated.

**Safeguards against AI error.** No synthetic data was used; only the supplied daily CSV. Tables and figures are produced by code that reads the saved models and results. Scripts check for data leakage and reload the saved models to reproduce the reported metrics. Claude ran its own checks in a Linux sandbox; those results are reference only and were not used as the reported results.

**Limits.** The AI could not run anything on my Mac. Its explanations of *why* the results look as they do are interpretations, not proven causes. I am responsible for understanding and being able to explain all of the code and text.

## 13. Appendix: verification logs and reference runs

In [ ]:
for title, path in [("Phase 2 leakage and integrity checks", PROCESSED_DIR / "leakage_verification.txt"), ("Phase 3 model and result checks", RESULTS_DIR / "phase3_verification.txt")]:
    print("=" * 90); print(title, "->", path.relative_to(ROOT)); print("=" * 90); print(path.read_text() if path.exists() else "(not found: run the verify script)")
shown = False
for name, label in [("linux_python311_run", "Linux, Python 3.11 (reference)"), ("python312_run_B", "Linux, Python 3.12 (reference)")]:
    p = ROOT / "reference_runs" / name / "results" / "metrics_all.csv"
    if not p.exists(): continue
    shown = True; ref = pd.read_csv(p).query("approach == 'return'").set_index(["model", "split"])["RMSE"]; mine = metrics_saved.query("approach == 'return'").set_index(["model", "split"])["RMSE"]
    comp = pd.DataFrame({"this run RMSE": mine, f"{label} RMSE": ref}); comp["difference"] = comp.iloc[:, 0] - comp.iloc[:, 1]
    print(f"--- vs {label} (reference only, not authoritative)"); display(comp.loc[[(LABELS[k], s) for k in keys for s in ["validation", "test"]]].round(4))
print("Notebook finished. Results were read from models/, results/ and data/processed/.")